In [ ]:
#import all libray i need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(data_path)

# we use read_csv to read the data and we put it in df_food

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_food.shape}")
df_food.head()

# we see overview of the data

In [ ]:
# Task 3: Write your code here:
# Check data types and structure
df_food.info()

In [ ]:
# Task 4: Write your code here:
# Descriptive statistics for numerical columns
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# plt distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean= df_food.drop(columns=['Order_ID'])
df_clean

In [ ]:
# Task 2: Write your code here:

# Count how many missing values exist in each column
# isnull() creates a boolean mask
# sum() counts how many True values (i.e., missing)
missing_values = df_clean.isnull().sum()
print("\nMissing values per column:")
print(missing_values)

# Percentage of missing values per column
missing_percentage = (missing_values / len(df_clean)) * 100
print("\nMissing values percentage:")
print(missing_percentage)


In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)



In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

print('data before encoding:\n', df_clean) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(df_clean) # Apply fit_transform

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

print('data before scaling:\n', data_onehot_encoded) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(data_onehot_encoded) # Apply fit_transform

print('\nData after scaling:\n', data_onehot_encoded) #show after scaling

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean['Delivery_Time']

# i make them agein to stop data leak

# check Do we have categorical columns?
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

label_encoders = encode_categorical_columns(df_clean)

from sklearn.preprocessing import LabelEncoder

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
df_clean

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()

kf = KFold(n_splits=5, shuffle=False)
scores = []

for tr, te in kf.split(X):

  Xtr, Xte = X.iloc[tr], X.iloc[te]
  ytr, yte = y.iloc[tr], y.iloc[te]
  model.fit(Xtr, ytr)
  scores.append(mean_squared_error(yte, model.predict (Xte)))
  print("K-Fold F1:", np.mean(scores))

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:
%pip install catboost

from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

# Define classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]), ('rf', models["Logistic Regression"]), ('gnb', models["Random Forest Classifier"])], voting="soft")

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")
